# 🚀 04: A100 Locked Testing and Serving Validation

*Part of the KV-Cache Eviction Capstone Series*
*Estimated time: 60–120 minutes (if executed at scale)*

---

Welcome to the final exam. This notebook is intentionally separated from development. We will load the frozen checkpoint selected in Notebook 03 and evaluate it on the 40 held-out controlled retrieval records spanning planned contexts of 1,024 through 8,192 tokens. An A100 may be used as the locked-test hardware, but this corpus does not justify a claim of 64k/128k evaluation.

### Experiment Roadmap
```text
[Frozen checkpoint] → [A100 preflight] → [Locked test] → [Observed artifacts]
```


## Step 1: Why Does This Matter?

A scientific result must be tested on unseen data using a frozen model. If we tweak the policy now, we invalidate the experiment.

**Locked-Test Contract:**
The checkpoint and hyperparameters are fixed. No test quality metric may drive model selection. If the runtime cannot support A100-scale contexts, the run is explicitly skipped with a structured status record.


In [ ]:
NOTEBOOK_ID = '04_a100_locked_test_serving'
REQUESTED_PROFILE = 'a100'

# Colab bootstrap: install pinned dependencies and unpack the shared core.
# Upload kvcore_bundle.zip supplied with this notebook suite if kvcore is not present.
from pathlib import Path
import sys, subprocess, zipfile

PINNED = [
    'transformers==4.56.2', 'accelerate==1.10.1', 'datasets==4.0.0',
    'huggingface_hub==0.34.4', 'bitsandbytes==0.47.0', 'safetensors==0.6.2',
    'sentencepiece==0.2.1', 'scipy==1.16.1', 'matplotlib==3.10.6',
    'seaborn==0.13.2', 'pandas==2.3.2',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PINNED])

if not Path('kvcore').exists():
    try:
        from google.colab import files
        print('Upload kvcore_bundle.zip from the delivered suite.')
        uploaded = files.upload()
        archive = next((Path(name) for name in uploaded if name.endswith('.zip')), None)
        if archive is None:
            raise FileNotFoundError('Please upload kvcore_bundle.zip.')
        with zipfile.ZipFile(archive) as zf:
            zf.extractall('.')
    except ImportError as error:
        raise RuntimeError('Run in Google Colab or place the kvcore directory beside this notebook.') from error

sys.path.insert(0, str(Path('.').resolve()))
from kvcore import *
from kvcore.config import BENCHMARKS, MODELS, POLICY_DEFAULTS, PROFILES, SUITE_VERSION
print({'suite_version': SUITE_VERSION, 'ruler_revision': BENCHMARKS['ruler']['revision'], 'longbench_revision': BENCHMARKS['longbench']['revision']})


In [ ]:
import json, os, platform, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

REQUESTED_PROFILE = REQUESTED_PROFILE  # defined by the notebook title cell
NOTEBOOK_ID = globals().get('NOTEBOOK_ID', 'runtime')
set_all_seeds(590)
profile, runtime_status = select_profile(REQUESTED_PROFILE)
run_root = ensure_run_root(f'kv_eviction_{NOTEBOOK_ID}')
manifest = run_manifest(
    run_root,
    notebook=NOTEBOOK_ID,
    requested_profile=REQUESTED_PROFILE,
    active_profile=profile.name,
    model=model_spec(profile.model_tier),
    runtime_status=runtime_status,
)
print(json.dumps({'notebook': NOTEBOOK_ID, 'requested_profile': REQUESTED_PROFILE, 'active_profile': profile.name, 'runtime_status': runtime_status, 'run_root': str(run_root)}, indent=2))
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU runtime is required for model execution. In Colab: Runtime > Change runtime type > GPU.')
print('GPU:', torch.cuda.get_device_name(0), 'VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))


In [ ]:
# Model and tokenizer are always loaded at immutable Hub revisions from kvcore.config.
# If the runtime is smaller than the requested tier, runtime_status records the fallback.
model, tokenizer = load_model_and_tokenizer(model_spec(profile.model_tier), load_mode=profile.load_mode, attn_implementation='sdpa')
print({'model': model.config._name_or_path, 'requested_profile': REQUESTED_PROFILE, 'active_profile': profile.name, 'load_mode': profile.load_mode})


In [ ]:
# Safe defaults: preflight and analytic accounting do not run the held-out test.
RUN_A100_TEST = False
RESUME_LOCKED_TEST = True
RUN_ANALYTIC_8BIT_ACCOUNTING = True
RUN_SERVING_MEASUREMENT = False

# Set these to the frozen Notebook 03 artifacts stored in Google Drive.
CHECKPOINT_PATH = None
SELECTION_RECORD_PATH = None

# When RUN_A100_TEST=True, progress is saved directly to this Drive directory
# after every policy-budget measurement. Keep this path unchanged when resuming.
LOCKED_TEST_RUN_ROOT = (
    '/content/drive/MyDrive/KV_Eviction_Capstone/02_notebook_runs/'
    '04_a100_locked_test_run/kv_eviction_04_a100_locked_test_serving'
)
artifact_root = run_root

print('Locked-test corpus:', CONTROLLED_RETRIEVAL['name'])
print('Planned context lengths:', CONTROLLED_RETRIEVAL['context_lengths'])
print('Held-out test records:', len(CONTROLLED_RETRIEVAL['context_lengths']) * CONTROLLED_RETRIEVAL['partition_counts']['test'])
print('Resume mode:', RESUME_LOCKED_TEST)


## Step 2: Preflight Scale and Cache Accounting

Before we allocate massive caches, we verify the actual runtime. This cell records an honest condition if the hardware cannot support the requested scale.


In [ ]:
from kvcore.evaluation import fp8_payload_accounting
preflight = {'requested_profile': 'a100', 'active_profile': profile.name, 'runtime_status': runtime_status, 'max_context_tokens': profile.max_context_tokens, 'dataset': CONTROLLED_RETRIEVAL['name'], 'planned_context_lengths': list(CONTROLLED_RETRIEVAL['context_lengths']), 'locked_test_records': 40}
if profile.name != 'a100':
    preflight['status'] = 'requires A100 — not run here'
    preflight['reason'] = 'Detected runtime does not meet the A100-scale profile.'
else:
    preflight['status'] = 'eligible_for_a100_run'
write_json(run_root / 'manifests' / 'a100_preflight.json', preflight)
print(preflight)


## Step 3: Locked-Test RULER Sweep

We now evaluate the full cache, the heuristics, and our frozen learned policy on the held-out test rows.


In [ ]:
if RUN_A100_TEST:
    from google.colab import drive
    import hashlib

    drive.mount('/content/drive')
    if CHECKPOINT_PATH is None or not Path(CHECKPOINT_PATH).exists():
        raise ValueError('Set CHECKPOINT_PATH to the existing frozen learned_kv_scorer.pt from Notebook 03.')
    if SELECTION_RECORD_PATH is None or not Path(SELECTION_RECORD_PATH).exists():
        raise ValueError('Set SELECTION_RECORD_PATH to the existing selected_by_validation_only.json from Notebook 03.')

    artifact_root = ensure_run_root(LOCKED_TEST_RUN_ROOT)
    learned, normaliser, payload = load_scorer(CHECKPOINT_PATH)
    assert_checkpoint_reload(learned, normaliser, CHECKPOINT_PATH, load_scorer)

    selected = json.loads(Path(SELECTION_RECORD_PATH).read_text())
    selected_checkpoint = Path(str(selected.get('checkpoint', '')))
    restored_checkpoint = Path(CHECKPOINT_PATH)
    if not selected_checkpoint.parts or len(restored_checkpoint.parts) < len(selected_checkpoint.parts) or restored_checkpoint.parts[-len(selected_checkpoint.parts):] != selected_checkpoint.parts:
        raise RuntimeError('The restored checkpoint does not match the relative checkpoint path in selected_by_validation_only.json; do not run the locked test.')

    all_rows = load_controlled_retrieval(tokenizer=tokenizer, seed=590)
    corpus_audit = audit_controlled_retrieval_corpus(all_rows)
    assert corpus_audit['passed'], corpus_audit
    test_rows = [row for row in all_rows if row['partition'] == 'test']
    assert len(test_rows) == 40, 'Locked test must retain all 40 preassigned test records.'

    build_benchmark_manifest(
        test_rows, artifact_root / 'manifests' / 'a100_locked_test_rows.json',
        benchmark_name='controlled_locked_test', seed=590,
    )
    profile_ids = tokenise_prompt(
        tokenizer, test_rows[0]['prompt'], profile.attention_profile_context_cap,
        next(model.parameters()).device,
    )
    meta = profile_short_context(
        model, profile_ids, min(profile_ids.shape[-1], profile.attention_profile_context_cap),
        artifact_root, 'a100_short_profile', profile.prefill_chunk_tokens,
    )
    profile_data = load_profile(meta['profile_path'], device=next(model.parameters()).device) if meta['attention_available'] else None
    specs = [
        {'type': 'full'}, {'type': 'fifo'}, {'type': 'uniform'}, {'type': 'sink_recent'},
        {'type': 'attention_topk'}, {'type': 'h2o'}, {'type': 'learned_block', 'block_size': 16},
    ]
    checkpoint_sha256 = hashlib.sha256(Path(CHECKPOINT_PATH).read_bytes()).hexdigest()
    resume_metadata = {
        'hardware': 'A100', 'checkpoint_path': str(CHECKPOINT_PATH),
        'selection_record_checkpoint': str(selected_checkpoint),
        'checkpoint_sha256': checkpoint_sha256, 'selection_record_path': str(SELECTION_RECORD_PATH),
        'selected_trial': selected.get('trial'), 'validation_ndcg_at_3': selected.get('validation_ndcg_at_3'),
        'model': model_spec(profile.model_tier), 'locked_test_rows': 40,
    }
    test_records = evaluate_rows(
        model, tokenizer, test_rows, specs, profile.budgets, profile.max_context_tokens,
        profile.decode_tokens, profile.prefill_chunk_tokens, artifact_root, profile=profile_data,
        learned_model=learned, normaliser=normaliser, artifact_stem='a100_controlled_locked_test',
        resume=RESUME_LOCKED_TEST, resume_metadata=resume_metadata,
    )
else:
    test_records = [requires_gpu_record('A100', 'Set RUN_A100_TEST=True with frozen checkpoint artifacts', 'Locked test execution is deliberately gated.')]

pd.DataFrame(test_records).to_csv(artifact_root / 'results' / 'a100_locked_test.csv', index=False)
print('Locked-test artifact root:', artifact_root)


## Step 4: Analytic 8-bit Accounting and Serving Boundary

This step records an **analytic 8-bit cache-payload estimate** and optional A100 serving evidence.

### What to Look For

The 8-bit estimate is explicitly labelled **analytic**. It is not an observed hardware-FP8 serving measurement on the A100. Measured latency and throughput require a separate serving backend and remain distinct from cache-policy simulation.

**Next step:** Save the persistent Drive run directory. We are ready for final aggregation in Notebook 05.

In [ ]:
if RUN_ANALYTIC_8BIT_ACCOUNTING:
    probe_ids = tokenizer('FP8 accounting probe. ' * 128, return_tensors='pt').input_ids.to(next(model.parameters()).device)
    probe_cache, _ = prefill(model, probe_ids, chunk_size=min(profile.prefill_chunk_tokens, probe_ids.shape[-1]), output_attentions=False)
    fp8 = fp8_payload_accounting(probe_cache)
    assert_fp8_byte_ratio(fp8)
    write_json(artifact_root / 'results' / 'analytic_8bit_accounting.json', fp8)
    print(fp8)

if RUN_SERVING_MEASUREMENT:
    print('Run the separate vLLM/SGLang service harness with the same model revision and workload manifest; export observed throughput/latency to results/serving_observed.csv.')
else:
    write_json(artifact_root / 'results' / 'serving_status.json', requires_gpu_record('A100 + serving backend', 'Set RUN_SERVING_MEASUREMENT=True', 'Serving is intentionally not inferred from analytic 8-bit accounting.'))


## Step 5: Pareto-Frontier Input Contract

This notebook prepares the artifacts for Pareto analysis. It does not label the frontier itself; that strict aggregation is reserved for Notebook 05.


In [ ]:
print('A100 notebook prepared. Artifact root:', artifact_root)

# Result tables only contain observed rows, explicit errors, or explicit hardware requirement statuses.
# No cell manufactures latency, memory, or quality values. Preserve run_root for the Results section.
